# Import 

In [ ]:
import pandas as pd 
import os 
import numpy as np
pd.set_option("display.float_format", "{:.0f}".format)
np.set_printoptions(precision=6)

os.chdir("D:/dataism/round 3.1/src") # Chỉnh lại đường dẫn chứa dự án
os.getcwd()

In [ ]:
df = pd.read_csv("data/dazone_sample.csv")
df.head()

In [ ]:
df['Unnamed: 0']

In [ ]:
print(f"df.shape: {df.shape}")

In [ ]:
df.drop(columns=["Unnamed: 0"], inplace=True)

In [ ]:
df.columns 

In [ ]:
col_name = "min_sign_vol"
df[col_name].isnull().mean() * 100 

In [ ]:
col_name = "min_sign_vol"
print(df[col_name].isnull().sum())
print(df[col_name].count())
print(len(df))
print(df[col_name].isnull().mean() * 100)

In [ ]:
def summarize_df(df): 
    null_count = df.isnull().sum()
    non_null_count = df.count()
    total_count = null_count + non_null_count

    null_pct = null_count / total_count * 100

    schema = pd.DataFrame({
        'Column'   : df.columns,
        'Dtype_raw': df.dtypes.values,
        'Non-Null' : non_null_count.values,
        'Null'     : null_count.values,
        'Null %'   : null_pct.map(lambda x: f"{x:.3f}").values,
        'Unique'   : df.nunique(dropna=True).values,
        'Sample'   : [
            str(df[c].dropna().iloc[0])[:40] if df[c].notna().any() else 'N/A'
            for c in df.columns
        ]
    })

    return schema

schema_df = summarize_df(df)
schema_df

Chia missing value như sau

| Missing % | Mức độ |
|-----------|---------|
| 0% | Không thiếu |
| (0%, 5%] | Rất ít |
| (5%, 20%] | Ít |
| (20%, 50%] | Trung bình |
| (50%, 80%] | Nhiều |
| > 80% | Rất nhiều / cân nhắc loại bỏ |

In [ ]:
def categorize_missing(ele): 
    ele = float(ele)

    if ele == 0: 
        cat = "no" 
    elif ele <= 5: 
        cat = "very_few" 
    elif ele <= 20: 
        cat = "few" 
    elif ele <= 50: 
        cat = "medium" 
    elif ele <= 80: 
        cat = "many" 
    else: 
        cat = "very_many" 

    return cat
    

schema_df["missing_type"] = schema_df["Null %"].map(categorize_missing)
schema_df

In [ ]:
missing_table = schema_df.groupby("missing_type")["Column"].apply(list).to_frame(name="column")
missing_table["count"] = missing_table["column"].map(len)
categories =[
    "no",
    "very_few",
    "few",
    "medium",
    "many",
    "very_many",
]
missing_table = missing_table.reindex(categories)
missing_table

In [ ]:
categories =[
    "no",
    "very_few",
    "few",
    "medium",
    "many",
    "very_many",
]

for cat in categories: 
    print(cat)
    print(missing_table.loc[cat, "column"])
    print(missing_table.loc[cat, "count"])
    print()

Các cột có tỉ lệ missing > 50% 

| Column | Meaning |
|----------|----------|
| `f_appointment_3m` | Flag nếu khách hàng có đặt lịch hẹn trong 3 tháng gần nhất với House Credit |
| `max_cntr_len_fi_12m` | Độ dài hợp đồng trả góp lớn nhất từ các tổ chức tài chính trong 12 tháng |
| `ratio_monthly_amt_l6m_vs_n6m` | Tỷ lệ số tiền phải trả hàng tháng với tất cả tổ chức và ngân hàng của 6 tháng gần nhất so với 6 tháng tiếp theo |
| `ratio_vol_hcvn_cash_appl_12m` | Tỷ lệ khoản tiền từ đơn vay tiền mặt House Credit trên tổng khoản tiền từ đơn vay tiền mặt ở tất cả tổ chức và ngân hàng trong lịch sử |
| `cnt_month_from_first_transaction` | Số tháng từ giao dịch chi tiêu đầu tiên với House Credit |
| `cnt_month_from_last_transaction` | Số tháng từ giao dịch chi tiêu gần nhất với House Credit |

In [ ]:
cat_cols = [
    "treatment",
    "cli_job",
    "cli_marital_status",
    "cli_contact_region_level",
    "cli_education",
    "last_process_channel",
    "risk_grp",
    "cli_contact_region_population_grp",
]

for col in cat_cols: 
    print(col)
    print(df[col].unique().tolist())
    print()

In [ ]:
nominal_cols = [
    'treatment', 
    'cli_job', 
    'cli_marital_status', 
    'cli_contact_region_level', 
    'last_process_channel', 
    'risk_grp', 
    'cli_contact_region_population_grp'
]

ordinal_cols = [
    'cli_education'
]

In [ ]:
num_cols = [
    "customer_id",
    "cash_sign_30d",
    "card_sign_30d",
    "min_sign_vol",
    "avg_sign_vol",
    "cnt_month_from_last_appl",
    "cli_age",
    "cli_emp_period",
    "last_offer_limit",
    "f_approve_last_process",
    "cnt_connected_tls",
    "cnt_fail_tls_3m",
    "f_appointment_3m",
    "cnt_total_payment",
    "ratio_monthly_amt_l6m_vs_n6m",
    "worst_credit_status",
    "cnt_month_status1",
    "ratio_vol_hcvn_cash_appl_12m",
    "f_ever_default",
    "ratio_conn_tls_weekend_12m",
    "ratio_conn_tls_endmonth_12m",
    "cnt_bank_inst_appl_12m",
    "cnt_fi_inst_appl_12m",
    "cnt_report_bank_6m",
    "cnt_report_institution_12m",
    "cnt_lender_has2pay_last_24m",
    "max_cntr_len_fi_12m",
    "amt_terminated_loan_12m",
    "cnt_terminated_loan_12m",
    "cnt_month_over3_credit_card_6m",
    "cnt_noinsurance_loan_24m",
    "cnt_earlyterminated_cash",
    "cnt_earlyterminted_pos",
    "cnt_inbound_tls_12m",
    "cnt_month_from_first_transaction",
    "cnt_month_from_last_transaction",
]

df[num_cols].describe().T

In [ ]:
for col in num_cols: 
    print(col)
    print()

In [ ]:
for col in num_cols: 
    print(col)
    print(len(df[col].unique()))
    print()

In [ ]:
num_cat_cols = [
    "cash_sign_30d",
    "card_sign_30d",
    "f_approve_last_process",
    "f_appointment_3m",
    "worst_credit_status",
    "f_ever_default",
    "cnt_report_bank_6m",
    "cnt_report_institution_12m",
    "cnt_lender_has2pay_last_24m",
    "cnt_terminated_loan_12m",
    "cnt_month_over3_credit_card_6m",
    "cnt_noinsurance_loan_24m",
    "cnt_earlyterminated_cash",
    "cnt_earlyterminted_pos",
    "cnt_inbound_tls_12m",
]

num_cont_cols = list(set(num_cols) - set(num_cat_cols))
num_cont_cols

In [ ]:
df.head()

In [ ]:
df.to_csv("data/cleaned_dazone_sample.csv", index=False)

# Import 

In [ ]:
import pandas as pd 
import os 
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
pd.reset_option("display.float_format")

os.chdir("D:/dataism/round 3.1/src")
os.getcwd()

# Đọc dữ liệu 

In [ ]:
df = pd.read_csv("data/cleaned_dazone_sample.csv")
df.head()

# Phân bố giá trị của các cột kiểu dữ liệu số liên tục  

In [ ]:
num_cont_cols = ['ratio_monthly_amt_l6m_vs_n6m',
 'amt_terminated_loan_12m',
 'cli_emp_period',
 'last_offer_limit',
 'max_cntr_len_fi_12m',
 'cnt_month_from_last_appl',
 'cnt_total_payment',
 'cnt_fail_tls_3m',
 'customer_id',
 'ratio_vol_hcvn_cash_appl_12m',
 'cnt_connected_tls',
 'cnt_month_from_first_transaction',
 'cnt_bank_inst_appl_12m',
 'cnt_fi_inst_appl_12m',
 'min_sign_vol',
 'cnt_month_from_last_transaction',
 'ratio_conn_tls_weekend_12m',
 'ratio_conn_tls_endmonth_12m',
 'avg_sign_vol',
 'cli_age',
 'cnt_month_status1']

df[num_cont_cols].describe().T

In [ ]:
n_cols = len(num_cont_cols)

fig, axes = plt.subplots(
    nrows=n_cols,
    ncols=2,
    figsize=(12, 4 * n_cols),
)

for i, col in enumerate(num_cont_cols):
    data = df[col].dropna()

    # Histogram
    sns.histplot(
        data,
        kde=True,
        ax=axes[i, 0],
    )
    axes[i, 0].set_title(f"{col} - Histogram")

    # Boxplot
    sns.boxplot(
        x=data,
        ax=axes[i, 1],
    )
    axes[i, 1].set_title(f"{col} - Boxplot")

plt.tight_layout()
plt.show()

## Sự tương quan giữa các cột

In [ ]:
corr = df[num_cont_cols].corr()

plt.figure(figsize=(14, 10))

sns.heatmap(
    corr,
    fmt=".2f",
    cmap="YlOrRd",       # Yellow -> Orange -> Red
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
)

plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## Các cột có sự tương quan cao 

In [ ]:
threshold = 0.7

corr = df[num_cont_cols].corr()

strong_pairs = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):

        col1 = corr.columns[i]
        col2 = corr.columns[j]

        corr_value = corr.iloc[i, j]

        if abs(corr_value) >= threshold:
            strong_pairs.append(
                (col1, col2, round(float(corr_value), 2))
            )

for item in strong_pairs: 
    print(item)

# Phân bố giá trị trong cột kiểu dữ liệu số rời rạc 

In [ ]:
num_cat_cols = [
    "cash_sign_30d",
    "card_sign_30d",
    "f_approve_last_process",
    "f_appointment_3m",
    "worst_credit_status",
    "f_ever_default",
    "cnt_report_bank_6m",
    "cnt_report_institution_12m",
    "cnt_lender_has2pay_last_24m",
    "cnt_terminated_loan_12m",
    "cnt_month_over3_credit_card_6m",
    "cnt_noinsurance_loan_24m",
    "cnt_earlyterminated_cash",
    "cnt_earlyterminted_pos",
    "cnt_inbound_tls_12m",
]



In [ ]:
fig, axes = plt.subplots(
    nrows=len(num_cat_cols),
    ncols=1,
    figsize=(10, 3 * len(num_cat_cols)),
)

if len(num_cat_cols) == 1:
    axes = [axes]

for i, col in enumerate(num_cat_cols):

    pct = (
        df[col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .sort_index()
    )

    ax = axes[i]

    sns.barplot(
        x=pct.values,
        y=pct.index.astype(str),
        ax=ax,
    )

    ax.set_title(col)
    ax.set_xlabel("Percentage (%)")
    ax.set_ylabel("")

    # Hiển thị giá trị %
    for j, value in enumerate(pct.values):
        ax.text(
            value + 0.5,
            j,
            f"{value:.2f}%",
            va="center",
        )

plt.tight_layout()
plt.show()

# Phân bố giá trị của các cột có kiểu dữ liệu phân loại 

In [ ]:
cat_cols = [
    'treatment', 
    'cli_job', 
    'cli_marital_status', 
    'cli_contact_region_level', 
    'last_process_channel', 
    'risk_grp', 
    'cli_contact_region_population_grp', 
    'cli_education', 
]

In [ ]:
fig, axes = plt.subplots(
    nrows=len(cat_cols),
    ncols=1,
    figsize=(10, 3 * len(cat_cols)),
)

if len(cat_cols) == 1:
    axes = [axes]

for i, col in enumerate(cat_cols):
    pct = (
        df[col]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .sort_index()
    )

    ax = axes[i]

    sns.barplot(
        x=pct.values,
        y=pct.index.astype(str),
        ax=ax,
    )

    ax.set_title(col)
    ax.set_xlabel("Percentage (%)")
    ax.set_ylabel("")

    # Hiển thị giá trị %
    for j, value in enumerate(pct.values):
        ax.text(
            value + 0.5,
            j,
            f"{value:.2f}%",
            va="center",
        )

plt.tight_layout()
plt.show()

# 1. Treatment Distribution

Bar chart số lượng khách hàng theo:

- Cash
- Card

In [ ]:
# đếm số lượng theo treatment
treatment_count = (
    df["treatment"]
    .value_counts()
    .reset_index()
)

treatment_count.columns = ["treatment", "count"]

# vẽ biểu đồ
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=treatment_count,
    x="treatment",
    y="count"
)

# hiển thị số lượng trên cột
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height()):,}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom"
    )

plt.title("Treatment Distribution")
plt.xlabel("Treatment")
plt.ylabel("Number of Customers")

plt.tight_layout()
plt.show()

**Nhận xét ngắn gọn:**

* Nhóm **Cash** có **75,612 khách hàng (≈55.2%)**, cao hơn nhóm **Card** với **61,380 khách hàng (≈44.8%)**.
* Phân bổ treatment **không hoàn toàn cân bằng 50-50**, chênh lệch khoảng **10.4 điểm phần trăm**.
* Mức chênh lệch này chưa quá nghiêm trọng, nhưng cần lưu ý khi đánh giá hiệu quả giữa hai chiến lược.
* Trong các bước tiếp theo, nên kiểm tra thêm xem các đặc trưng quan trọng (tuổi, risk_grp, region, lịch sử tín dụng,...) có phân bố tương đồng giữa hai nhóm hay không để xác nhận tính ngẫu nhiên của treatment.

**Business insight:**

> House Credit đã phân bổ nhiều khách hàng vào chiến lược Cash hơn Card. Trước khi xây dựng uplift model, cần xác minh rằng sự khác biệt về số lượng không đi kèm với khác biệt lớn về đặc điểm khách hàng giữa hai nhóm. Nếu không, kết quả uplift có thể bị ảnh hưởng bởi selection bias thay vì hiệu ứng thực sự của chiến lược Cash/Card.


# 2. Conversion Rate theo Treatment

Bar chart:

- Cash → tỷ lệ cash_sign_30d
- Card → tỷ lệ card_sign_30d

In [ ]:
# conversion rate theo treatment
conversion_rate = pd.DataFrame({
    "treatment": ["cash", "card"],
    "conversion_rate": [
        df.loc[df["treatment"] == "cash", "cash_sign_30d"].mean() * 100,
        df.loc[df["treatment"] == "card", "card_sign_30d"].mean() * 100,
    ]
})

plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=conversion_rate,
    x="treatment",
    y="conversion_rate"
)

# hiện % trên cột
for p in ax.patches:
    ax.annotate(
        f"{p.get_height():.2f}%",
        (
            p.get_x() + p.get_width()/2,
            p.get_height()
        ),
        ha="center",
        va="bottom"
    )

plt.title("Conversion Rate by Treatment")
plt.xlabel("Treatment")
plt.ylabel("Conversion Rate (%)")

plt.tight_layout()
plt.show()

**Nhận xét ngắn gọn:**

* Nhóm được truyền thông **Cash Loan** có tỷ lệ chuyển đổi **6.26%**, cao hơn đáng kể so với nhóm **Credit Card** (**2.84%**).
* Chênh lệch đạt khoảng **3.42 điểm phần trăm**, tương đương tỷ lệ chuyển đổi của Cash cao gấp khoảng **2.2 lần** Card.
* Ở mức tổng thể, chiến lược **Cash Loan đang cho hiệu quả chuyển đổi tốt hơn Credit Card** trên tập dữ liệu hiện tại.

**Lưu ý business quan trọng:**

Biểu đồ này chỉ là **baseline quan sát được**, chưa thể kết luận rằng "Cash tốt hơn Card cho mọi khách hàng". Mục tiêu của uplift modeling là tìm ra:

> Khách hàng nào nên nhận Cash và khách hàng nào nên nhận Card,

vì có thể tồn tại những phân khúc nhỏ mà Credit Card mang lại hiệu quả cao hơn dù trung bình toàn bộ tập dữ liệu thấp hơn. Đây chính là lý do cần tiếp tục phân tích theo các nhóm như `risk_grp`, `cli_age`, `region`, `education`,... trước khi xây dựng mô hình uplift.


# 3. Cross Conversion Matrix
**Ý nghĩa business**

Đo mức độ khách hàng vẫn đăng ký sản phẩm còn lại dù nhận treatment nào.

cho code python vẽ biểu đồ trên

In [ ]:
# Tính conversion rate (%)
cross_matrix = pd.DataFrame({
    "Cash Sign": [
        df.loc[df["treatment"] == "cash", "cash_sign_30d"].mean() * 100,
        df.loc[df["treatment"] == "card", "cash_sign_30d"].mean() * 100,
    ],
    "Card Sign": [
        df.loc[df["treatment"] == "cash", "card_sign_30d"].mean() * 100,
        df.loc[df["treatment"] == "card", "card_sign_30d"].mean() * 100,
    ]
},
index=["Cash Treatment", "Card Treatment"])

plt.figure(figsize=(8, 5))

sns.heatmap(
    cross_matrix,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Conversion Rate (%)"}
)

plt.title("Cross Conversion Matrix")
plt.xlabel("Outcome")
plt.ylabel("Treatment")

plt.tight_layout()
plt.show()

**Nhận xét ngắn gọn:**

* Dù được truyền thông **Credit Card**, khách hàng vẫn đăng ký **Cash Loan với tỷ lệ 5.50%**, cao gần gấp đôi tỷ lệ mở **Credit Card (2.84%)**.
* Điều này cho thấy **nhu cầu Cash Loan tự nhiên mạnh hơn Credit Card** trong tập khách hàng hiện tại.
* Hiệu quả của chiến lược Card tương đối hạn chế: nhiều khách hàng nhận Card nhưng cuối cùng vẫn chọn Cash.

**Business insight:**

> Chỉ riêng việc chuyển từ Card sang Cash có thể mang lại mức chuyển đổi cao hơn cho nhiều khách hàng.

Đây là tín hiệu rất quan trọng cho uplift modeling: mô hình cần tìm ra **những phân khúc hiếm hoi mà Credit Card thực sự hiệu quả hơn Cash**, thay vì mặc định gửi Cash cho toàn bộ khách hàng.


# 4. Age vs Conversion × Treatment

Line chart hoặc bar chart theo age bucket có tách theo treatment:

- <25
- 25-35
- 35-45
- 45-55
- 55

In [ ]:

# tạo age bucket
df["age_bucket"] = pd.cut(
    df["cli_age"],
    bins=[0, 25, 35, 45, 55, np.inf],
    labels=["<25", "25-35", "35-45", "45-55", "55+"],
    right=False
)

# tính conversion đúng theo treatment
cash_age = (
    df[df["treatment"] == "cash"]
    .groupby("age_bucket")["cash_sign_30d"]
    .mean()
    .mul(100)
)

card_age = (
    df[df["treatment"] == "card"]
    .groupby("age_bucket")["card_sign_30d"]
    .mean()
    .mul(100)
)

# gom lại để vẽ
plot_df = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_age,
    "Card Treatment → Card Sign": card_age
}).reset_index()

plot_df = plot_df.melt(
    id_vars="age_bucket",
    var_name="treatment_outcome",
    value_name="conversion_rate"
)

# vẽ bar chart
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=plot_df,
    x="age_bucket",
    y="conversion_rate",
    hue="treatment_outcome"
)

for p in ax.patches:
    height = p.get_height()
    if not np.isnan(height):
        ax.annotate(
            f"{height:.2f}%",
            (p.get_x() + p.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.title("Conversion Rate by Age Bucket and Treatment")
plt.xlabel("Age Bucket")
plt.ylabel("Conversion Rate (%)")
plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

**Nhận xét ngắn gọn:**

* Trong **tất cả các nhóm tuổi**, chiến lược **Cash Loan vượt trội hơn Credit Card** về tỷ lệ chuyển đổi.
* Nhóm **55+** có mức phản hồi với Cash cao nhất (**8.74%**), trong khi Card chỉ đạt **2.14%**.
* Nhóm **<25 tuổi** có tỷ lệ chuyển đổi Credit Card cao nhất (**4.47%**), nhưng vẫn thấp hơn Cash (**7.66%**).
* Chênh lệch giữa Cash và Card ngày càng lớn ở các nhóm tuổi cao hơn.

**Business Insight**

* **Cash Loan là chiến lược mạnh hơn trên toàn bộ các nhóm tuổi** trong dữ liệu hiện tại.
* **Khách hàng trẻ (<25)** là phân khúc có tiềm năng nhất cho Credit Card vì tỷ lệ phản hồi Card cao nhất trong các nhóm tuổi.
* **Khách hàng lớn tuổi (55+)** rất phù hợp với Cash Loan và ít phản hồi với Credit Card.
* Tuổi (`cli_age`) là một feature có giá trị cho uplift model vì nó giúp phân biệt mức độ hiệu quả tương đối của Cash và Card giữa các phân khúc khách hàng.

**Góc nhìn uplift**

Nếu tính nhanh:

| Age Bucket |  Cash |  Card | Chênh lệch |
| ---------- | ----: | ----: | ---------: |
| <25        | 7.66% | 4.47% |     +3.19% |
| 25-35      | 6.28% | 3.05% |     +3.23% |
| 35-45      | 5.89% | 2.36% |     +3.53% |
| 45-55      | 5.52% | 2.11% |     +3.41% |
| 55+        | 8.74% | 2.14% |     +6.60% |

=> Nhóm **55+** hiện đang có **uplift Cash vs Card lớn nhất**, là phân khúc mà việc ưu tiên Cash có thể mang lại giá trị kinh doanh cao nhất.


In [ ]:
plot_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_age,
    "Card Treatment → Card Sign": card_age,
})

plot_table["uplift_cash_vs_card"] = (
    plot_table["Cash Treatment → Cash Sign"]
    - plot_table["Card Treatment → Card Sign"]
)

plot_table

# 5. Job Group vs Conversion

Horizontal bar chart có tách theo treatment:

x = nghề nghiệp

y = conversion rate

In [ ]:

# Cash treatment -> Cash sign
cash_job = (
    df[df["treatment"] == "cash"]
    .groupby("cli_job")["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> Card sign
card_job = (
    df[df["treatment"] == "card"]
    .groupby("cli_job")["card_sign_30d"]
    .mean()
    .mul(100)
)

# tạo bảng
job_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_job,
    "Card Treatment → Card Sign": card_job,
})

job_table["uplift_cash_vs_card"] = (
    job_table["Cash Treatment → Cash Sign"]
    - job_table["Card Treatment → Card Sign"]
)

# sắp xếp theo uplift
job_table = job_table.sort_values(
    "uplift_cash_vs_card",
    ascending=False
)

# long format để vẽ
plot_df = (
    job_table[
        [
            "Cash Treatment → Cash Sign",
            "Card Treatment → Card Sign"
        ]
    ]
    .reset_index()
    .melt(
        id_vars="cli_job",
        var_name="treatment",
        value_name="conversion_rate"
    )
)

# vẽ
plt.figure(figsize=(12, 8))

ax = sns.barplot(
    data=plot_df,
    y="cli_job",
    x="conversion_rate",
    hue="treatment",
    orient="h"
)

for p in ax.patches:
    width = p.get_width()

    if not np.isnan(width):
        ax.annotate(
            f"{width:.2f}%",
            (
                width,
                p.get_y() + p.get_height()/2
            ),
            va="center",
            ha="left",
            fontsize=8
        )

plt.title("Conversion Rate by Job Group and Treatment")
plt.xlabel("Conversion Rate (%)")
plt.ylabel("Job Group")

plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

In [ ]:
job_table.round(2)

**Nhận xét ngắn gọn:**

* Tất cả các nhóm nghề trong bảng đều có **uplift dương**, tức là **Cash Loan hiệu quả hơn Credit Card**.

* Nhóm **WORKER** có:

  * Conversion Cash cao nhất (**7.99%**)
  * Uplift lớn nhất (**+4.74 điểm %**)

  ⇒ Đây là nhóm phản ứng mạnh nhất với chiến lược Cash.

* Nhóm **ENGINEER** cũng có uplift khá cao (**+3.84 điểm %**), cho thấy Cash vượt trội rõ rệt so với Card.

* Nhóm **SERVICES**, **ADMINISTRATIVE** và **XNA** có tỷ lệ chuyển đổi Card tương đối cao hơn các nhóm khác, nhưng Cash vẫn thắng.

**Business Insight**

* Nếu ngân sách marketing hạn chế, các nhóm **WORKER**, **ENGINEER**, **SALESMAN** là ứng viên ưu tiên cho chiến dịch Cash Loan.
* Hiện chưa thấy nhóm nghề nào mà Credit Card vượt Cash.
* `cli_job` có khả năng là feature quan trọng trong uplift model vì mức chênh lệch Cash–Card khác nhau đáng kể giữa các nghề nghiệp.



# 7. Education vs Conversion

In [ ]:
# Cash treatment -> Cash sign
cash_edu = (
    df[df["treatment"] == "cash"]
    .groupby("cli_education")["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> Card sign
card_edu = (
    df[df["treatment"] == "card"]
    .groupby("cli_education")["card_sign_30d"]
    .mean()
    .mul(100)
)

# tạo bảng
edu_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_edu,
    "Card Treatment → Card Sign": card_edu,
})

edu_table["uplift_cash_vs_card"] = (
    edu_table["Cash Treatment → Cash Sign"]
    - edu_table["Card Treatment → Card Sign"]
)

# sắp xếp theo trình độ học vấn (nếu muốn)
edu_order = [
    "ELEMENTARY_SCHOOL",
    "PTCS",
    "PTTH",
    "BACHELORS",
    "MASTERS"
]

edu_table = (
    edu_table
    .reindex([x for x in edu_order if x in edu_table.index])
)

# long format
plot_df = (
    edu_table[
        [
            "Cash Treatment → Cash Sign",
            "Card Treatment → Card Sign"
        ]
    ]
    .reset_index()
    .melt(
        id_vars="cli_education",
        var_name="treatment",
        value_name="conversion_rate"
    )
)

# vẽ
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=plot_df,
    x="cli_education",
    y="conversion_rate",
    hue="treatment"
)

for p in ax.patches:
    height = p.get_height()

    if not np.isnan(height):
        ax.annotate(
            f"{height:.2f}%",
            (
                p.get_x() + p.get_width()/2,
                height
            ),
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.title("Conversion Rate by Education and Treatment")
plt.xlabel("Education")
plt.ylabel("Conversion Rate (%)")

plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

In [ ]:
edu_table.round(2)

- Tất cả các nhóm học vấn đều có uplift dương, nghĩa là Cash Loan đang cho tỷ lệ chuyển đổi cao hơn Credit Card ở mọi cấp độ học vấn.

- Nhóm PTTH có hiệu quả Cash Loan cao nhất (7.16%) và cũng có uplift lớn nhất (+4.03 điểm %), cho thấy đây là phân khúc phản ứng rất tốt với chiến dịch Cash.

- Nhóm MASTERS có tỷ lệ chuyển đổi Credit Card cao nhất (4.78%) và uplift thấp nhất (+1.98 điểm %), cho thấy Credit Card cạnh tranh tốt hơn ở nhóm học vấn cao.

- Khoảng cách giữa Cash và Card có xu hướng thu hẹp khi trình độ học vấn tăng lên:
  - ELEMENTARY_SCHOOL: +3.02%
  - PTCS: +3.51%
  - PTTH: +4.03%
  - BACHELORS: +2.13%
  - MASTERS: +1.98%

- Kết quả này ủng hộ giả thuyết business rằng khách hàng có học vấn cao có xu hướng quan tâm Credit Card nhiều hơn so với các nhóm học vấn thấp.

- Trong uplift modeling, `cli_education` là feature đáng chú ý vì có khả năng giúp nhận diện các phân khúc mà Credit Card có cơ hội cạnh tranh với Cash Loan.

# 8. Region vs Conversion

In [ ]:

# Cash treatment -> Cash sign
cash_region = (
    df[df["treatment"] == "cash"]
    .groupby("cli_contact_region_level")["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> Card sign
card_region = (
    df[df["treatment"] == "card"]
    .groupby("cli_contact_region_level")["card_sign_30d"]
    .mean()
    .mul(100)
)

# tạo bảng
region_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_region,
    "Card Treatment → Card Sign": card_region,
})

region_table["uplift_cash_vs_card"] = (
    region_table["Cash Treatment → Cash Sign"]
    - region_table["Card Treatment → Card Sign"]
)

# sắp xếp đúng thứ tự vùng nếu có
region_order = ["North", "Central", "South"]

region_table = region_table.reindex(
    [x for x in region_order if x in region_table.index]
)

# long format để vẽ
plot_df = (
    region_table[
        [
            "Cash Treatment → Cash Sign",
            "Card Treatment → Card Sign"
        ]
    ]
    .reset_index()
    .melt(
        id_vars="cli_contact_region_level",
        var_name="treatment",
        value_name="conversion_rate"
    )
)

# vẽ bar chart
plt.figure(figsize=(9, 6))

ax = sns.barplot(
    data=plot_df,
    x="cli_contact_region_level",
    y="conversion_rate",
    hue="treatment"
)

# hiện % trên cột
for p in ax.patches:
    height = p.get_height()

    if not np.isnan(height):
        ax.annotate(
            f"{height:.2f}%",
            (
                p.get_x() + p.get_width() / 2,
                height
            ),
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.title("Conversion Rate by Region and Treatment")
plt.xlabel("Region")
plt.ylabel("Conversion Rate (%)")

plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

In [ ]:
region_table.round(2)

- Cả 3 vùng đều có uplift dương, cho thấy Cash Loan đang mang lại tỷ lệ chuyển đổi cao hơn Credit Card trên toàn quốc.

- South là khu vực có hiệu quả cao nhất:
  - Cash: 7.08%
  - Card: 3.34%
  - Uplift: +3.74 điểm %

- North đứng thứ hai về hiệu quả:
  - Cash: 5.76%
  - Card: 2.26%
  - Uplift: +3.50 điểm %

- Central có tỷ lệ chuyển đổi thấp nhất cho cả hai sản phẩm:
  - Cash: 4.95%
  - Card: 2.39%
  - Uplift: +2.55 điểm %

- South không chỉ có tỷ lệ chuyển đổi cao nhất mà còn có khoảng cách Cash–Card lớn nhất, cho thấy khách hàng khu vực này phản ứng đặc biệt tốt với Cash Loan.

- Chưa có khu vực nào mà Credit Card vượt hoặc tiệm cận Cash Loan, tuy nhiên Central là vùng có uplift thấp nhất nên có thể là nơi Card có khả năng cạnh tranh tốt hơn tương đối so với các vùng còn lại.

- `cli_contact_region_level` là feature hữu ích cho uplift model vì hành vi phản hồi giữa các vùng địa lý có sự khác biệt đáng kể.

# 9. Urbanization vs Conversion

In [ ]:

# Cash treatment -> Cash sign
cash_urban = (
    df[df["treatment"] == "cash"]
    .groupby("cli_contact_region_population_grp")["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> Card sign
card_urban = (
    df[df["treatment"] == "card"]
    .groupby("cli_contact_region_population_grp")["card_sign_30d"]
    .mean()
    .mul(100)
)

# tạo bảng
urban_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_urban,
    "Card Treatment → Card Sign": card_urban,
})

urban_table["uplift_cash_vs_card"] = (
    urban_table["Cash Treatment → Cash Sign"]
    - urban_table["Card Treatment → Card Sign"]
)

# sắp xếp theo mức độ đô thị hóa
urban_order = [
    "KEY_URBAN",
    "URBAN",
    "SUB_URBAN",
    "RURAL"
]

urban_table = urban_table.reindex(
    [x for x in urban_order if x in urban_table.index]
)

# long format
plot_df = (
    urban_table[
        [
            "Cash Treatment → Cash Sign",
            "Card Treatment → Card Sign"
        ]
    ]
    .reset_index()
    .melt(
        id_vars="cli_contact_region_population_grp",
        var_name="treatment",
        value_name="conversion_rate"
    )
)

# vẽ biểu đồ
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=plot_df,
    x="cli_contact_region_population_grp",
    y="conversion_rate",
    hue="treatment"
)

# hiện giá trị %
for p in ax.patches:
    height = p.get_height()

    if not np.isnan(height):
        ax.annotate(
            f"{height:.2f}%",
            (
                p.get_x() + p.get_width()/2,
                height
            ),
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.title("Conversion Rate by Urbanization Level and Treatment")
plt.xlabel("Urbanization Level")
plt.ylabel("Conversion Rate (%)")

plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

In [ ]:
urban_table.round(2)

- Cả 4 mức độ đô thị hóa đều có uplift dương, cho thấy Cash Loan đang mang lại tỷ lệ chuyển đổi cao hơn Credit Card ở mọi khu vực.

- SUB_URBAN có hiệu quả cao nhất:
  - Cash: 6.87%
  - Card: 2.92%
  - Uplift: +3.96 điểm %

- KEY_URBAN có tỷ lệ chuyển đổi Credit Card cao nhất (3.12%), cho thấy khách hàng ở khu vực đô thị trọng điểm có xu hướng tiếp nhận sản phẩm thẻ tốt hơn các khu vực khác.

- URBAN có uplift thấp nhất (+2.95 điểm %), nghĩa là khoảng cách giữa Cash và Card nhỏ nhất trong các nhóm.

- RURAL vẫn phản ứng tốt với Cash Loan (6.03%), trong khi Credit Card thấp hơn đáng kể (2.72%).

- Kết quả cho thấy mức độ đô thị hóa có ảnh hưởng đến hiệu quả của hai chiến lược, đặc biệt là khả năng tiếp nhận Credit Card ở các khu vực đô thị.

- `cli_contact_region_population_grp` là feature có giá trị cho uplift model vì phản ánh khác biệt hành vi tài chính giữa khách hàng ở khu vực đô thị và nông thôn.

# 10. Risk Group vs Conversion

In [ ]:

# Cash treatment -> Cash sign
cash_risk = (
    df[df["treatment"] == "cash"]
    .groupby("risk_grp")["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> Card sign
card_risk = (
    df[df["treatment"] == "card"]
    .groupby("risk_grp")["card_sign_30d"]
    .mean()
    .mul(100)
)

# tạo bảng
risk_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_risk,
    "Card Treatment → Card Sign": card_risk,
})

risk_table["uplift_cash_vs_card"] = (
    risk_table["Cash Treatment → Cash Sign"]
    - risk_table["Card Treatment → Card Sign"]
)

# sort theo risk group
risk_table = risk_table.sort_index()

# vẽ line chart
plt.figure(figsize=(10, 6))

plt.plot(
    risk_table.index,
    risk_table["Cash Treatment → Cash Sign"],
    marker="o",
    linewidth=2,
    label="Cash Treatment → Cash Sign"
)

plt.plot(
    risk_table.index,
    risk_table["Card Treatment → Card Sign"],
    marker="o",
    linewidth=2,
    label="Card Treatment → Card Sign"
)

# hiện giá trị trên điểm
for x, y in zip(
    risk_table.index,
    risk_table["Cash Treatment → Cash Sign"]
):
    plt.annotate(
        f"{y:.2f}%",
        (x, y),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center"
    )

for x, y in zip(
    risk_table.index,
    risk_table["Card Treatment → Card Sign"]
):
    plt.annotate(
        f"{y:.2f}%",
        (x, y),
        textcoords="offset points",
        xytext=(0, -15),
        ha="center"
    )

plt.title("Conversion Rate by Risk Group and Treatment")
plt.xlabel("Risk Group")
plt.ylabel("Conversion Rate (%)")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
risk_table.round(2)

- Nhóm rủi ro từ A → E cho thấy xu hướng rất rõ: khi mức rủi ro tăng lên, tỷ lệ chuyển đổi của cả Cash Loan và Credit Card đều giảm dần.

- Nhóm A có hiệu quả cao nhất:
  - Cash: 13.76%
  - Card: 6.70%
  - Uplift: +7.07 điểm %

- Nhóm F là điểm bất thường:
  - Cash: 14.29%
  - Card: 3.70%
  - Uplift: +10.58 điểm %
  
  Cần kiểm tra thêm số lượng quan sát của nhóm F vì có thể đây là nhóm rất nhỏ hoặc có đặc điểm đặc biệt.

- Nhóm G và H có tỷ lệ chuyển đổi bằng 0 cho cả hai sản phẩm, khả năng cao số lượng khách hàng rất ít hoặc không phát sinh chuyển đổi.

- Nhóm I là một ngoại lệ lớn:
  - Cash: 0%
  - Card: 33.33%
  - Uplift: -33.33 điểm %
  
  Cần kiểm tra số lượng mẫu trước khi kết luận vì kết quả này có thể đến từ rất ít khách hàng.

- Đối với các nhóm có đủ dữ liệu (A–F), Cash Loan luôn vượt Credit Card và khoảng cách khá lớn.

- Kết quả cho thấy `risk_grp` là một trong những feature business mạnh nhất trong bộ dữ liệu vì khả năng phân tách hành vi khách hàng rất rõ theo mức độ rủi ro.

- Trước khi đưa ra insight cuối cùng, nên kiểm tra thêm số lượng khách hàng của từng `risk_grp` bằng:

```python
df["risk_grp"].value_counts()
```

vì các nhóm G, H, I có dấu hiệu dữ liệu rất ít và có thể gây nhiễu khi phân tích uplift.

# 11. Worst Credit Status vs Conversion

In [ ]:
# Cash treatment -> Cash sign
cash_credit = (
    df[df["treatment"] == "cash"]
    .groupby("worst_credit_status")["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> Card sign
card_credit = (
    df[df["treatment"] == "card"]
    .groupby("worst_credit_status")["card_sign_30d"]
    .mean()
    .mul(100)
)

# tạo bảng
credit_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_credit,
    "Card Treatment → Card Sign": card_credit,
})

credit_table["uplift_cash_vs_card"] = (
    credit_table["Cash Treatment → Cash Sign"]
    - credit_table["Card Treatment → Card Sign"]
)

credit_table = credit_table.sort_index()

# vẽ line chart
plt.figure(figsize=(10, 6))

plt.plot(
    credit_table.index,
    credit_table["Cash Treatment → Cash Sign"],
    marker="o",
    linewidth=2,
    label="Cash Treatment → Cash Sign"
)

plt.plot(
    credit_table.index,
    credit_table["Card Treatment → Card Sign"],
    marker="o",
    linewidth=2,
    label="Card Treatment → Card Sign"
)

# hiển thị giá trị
for x, y in zip(
    credit_table.index,
    credit_table["Cash Treatment → Cash Sign"]
):
    plt.annotate(
        f"{y:.2f}%",
        (x, y),
        textcoords="offset points",
        xytext=(0, 8),
        ha="center"
    )

for x, y in zip(
    credit_table.index,
    credit_table["Card Treatment → Card Sign"]
):
    plt.annotate(
        f"{y:.2f}%",
        (x, y),
        textcoords="offset points",
        xytext=(0, -15),
        ha="center"
    )

plt.title("Conversion Rate by Worst Credit Status and Treatment")
plt.xlabel("Worst Credit Status")
plt.ylabel("Conversion Rate (%)")

plt.xticks([1, 2, 3, 4, 5, 6])

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
credit_table.round(2)

- Nhóm tín dụng tốt (status 1 và 2) có tỷ lệ chuyển đổi cao nhất cho cả hai sản phẩm, cho thấy lịch sử tín dụng là yếu tố ảnh hưởng mạnh đến khả năng phản hồi của khách hàng.

- Nhóm status 2 có hiệu quả cao nhất:
  - Cash: 11.58%
  - Card: 4.41%
  - Uplift: +7.17 điểm %

- Khi chất lượng tín dụng xấu đi (status 3 → 5), tỷ lệ chuyển đổi giảm rất mạnh đối với cả Cash Loan và Credit Card.

- Status 3 là nhóm duy nhất có uplift âm:
  - Cash: 1.01%
  - Card: 1.39%
  - Uplift: -0.38 điểm %
  
  Đây là tín hiệu hiếm hoi cho thấy Credit Card có thể hiệu quả hơn Cash Loan ở một phân khúc khách hàng cụ thể.

- Các nhóm status 4, 5, 6 có tỷ lệ chuyển đổi rất thấp (<4%), cho thấy khả năng tiếp nhận sản phẩm tài chính hạn chế ở các khách hàng có lịch sử tín dụng xấu.

- Nhóm status 6 (Write-off) có tỷ lệ chuyển đổi cao hơn status 4 và 5, đây là dấu hiệu bất thường và cần kiểm tra thêm số lượng khách hàng trong từng nhóm trước khi đưa ra kết luận business.

- Kết quả cho thấy `worst_credit_status` là feature rất quan trọng cho uplift model vì có khả năng phân biệt rõ các phân khúc khách hàng phản ứng khác nhau với Cash và Card.

- Đặc biệt, status 3 là nhóm cần theo dõi kỹ vì hiện tại đây là ứng viên hiếm hoi mà Credit Card đang có dấu hiệu cạnh tranh hoặc vượt Cash Loan.

# 12. Ever Default vs Conversion

In [ ]:
# Cash treatment -> Cash sign
cash_default = (
    df[df["treatment"] == "cash"]
    .groupby("f_ever_default")["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> Card sign
card_default = (
    df[df["treatment"] == "card"]
    .groupby("f_ever_default")["card_sign_30d"]
    .mean()
    .mul(100)
)

# tạo bảng
default_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_default,
    "Card Treatment → Card Sign": card_default,
})

default_table["uplift_cash_vs_card"] = (
    default_table["Cash Treatment → Cash Sign"]
    - default_table["Card Treatment → Card Sign"]
)

# đổi tên cho dễ đọc
default_table.index = [
    "Non-Default",
    "Default"
]

# long format
plot_df = (
    default_table[
        [
            "Cash Treatment → Cash Sign",
            "Card Treatment → Card Sign"
        ]
    ]
    .reset_index(names="default_status")
    .melt(
        id_vars="default_status",
        var_name="treatment",
        value_name="conversion_rate"
    )
)

# vẽ
plt.figure(figsize=(8, 6))

ax = sns.barplot(
    data=plot_df,
    x="default_status",
    y="conversion_rate",
    hue="treatment"
)

# hiện giá trị %
for p in ax.patches:
    height = p.get_height()

    if not np.isnan(height):
        ax.annotate(
            f"{height:.2f}%",
            (
                p.get_x() + p.get_width()/2,
                height
            ),
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.title("Conversion Rate by Ever Default Status")
plt.xlabel("Default History")
plt.ylabel("Conversion Rate (%)")

plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

In [ ]:
default_table.round(2)

- Kết quả cho thấy nhóm khách hàng từng có lịch sử default lại có tỷ lệ chuyển đổi cao hơn đáng kể so với nhóm chưa từng default đối với cả hai sản phẩm.

- Nhóm Default:
  - Cash: 11.61%
  - Card: 4.41%
  - Uplift: +7.20 điểm %

- Nhóm Non-Default:
  - Cash: 5.81%
  - Card: 2.70%
  - Uplift: +3.11 điểm %

- Cash Loan vượt Credit Card ở cả hai nhóm, nhưng khoảng cách ở nhóm Default lớn hơn rất nhiều.

- Đây là kết quả khá bất thường về mặt business vì thông thường khách hàng từng nợ xấu sẽ có tỷ lệ chuyển đổi thấp hơn. Điều này có thể phản ánh:
  - Nhóm Default trong dữ liệu hiện tại là những khách hàng đã quay lại và vẫn đang được House Credit chủ động tiếp cận.
  - Hoặc số lượng khách hàng Default tương đối nhỏ và có đặc điểm khác biệt với phần còn lại của tập dữ liệu.

- Cần kiểm tra thêm số lượng khách hàng của từng nhóm:

```python
df["f_ever_default"].value_counts()
```

- Nếu số lượng đủ lớn, đây là insight rất giá trị: khách hàng từng default có vẻ phản ứng đặc biệt mạnh với Cash Loan và có thể là một phân khúc tiềm năng cho chiến lược Cash.

# 13. Offer Limit vs Conversion

In [ ]:
df["offer_limit_decile"] = pd.qcut(
    df["last_offer_limit"],
    q=10,
    labels=[f"D{i}" for i in range(1, 11)],
    duplicates="drop"
)

cash_limit = (
    df[df["treatment"] == "cash"]
    .groupby("offer_limit_decile")["cash_sign_30d"]
    .mean()
    .mul(100)
)

card_limit = (
    df[df["treatment"] == "card"]
    .groupby("offer_limit_decile")["card_sign_30d"]
    .mean()
    .mul(100)
)

limit_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_limit,
    "Card Treatment → Card Sign": card_limit,
})

limit_table["uplift_cash_vs_card"] = (
    limit_table["Cash Treatment → Cash Sign"]
    - limit_table["Card Treatment → Card Sign"]
)

plot_df = (
    limit_table[
        [
            "Cash Treatment → Cash Sign",
            "Card Treatment → Card Sign"
        ]
    ]
    .reset_index()
    .melt(
        id_vars="offer_limit_decile",
        var_name="treatment",
        value_name="conversion_rate"
    )
)

plt.figure(figsize=(12, 6))

ax = sns.barplot(
    data=plot_df,
    x="offer_limit_decile",
    y="conversion_rate",
    hue="treatment"
)

for p in ax.patches:
    height = p.get_height()

    if not np.isnan(height):
        ax.annotate(
            f"{height:.2f}%",
            (
                p.get_x() + p.get_width()/2,
                height
            ),
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.title("Conversion Rate by Offer Limit Decile")
plt.xlabel("Offer Limit Decile")
plt.ylabel("Conversion Rate (%)")

plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

In [ ]:
limit_table.round(2)

- Không quan sát thấy xu hướng tuyến tính rõ ràng rằng hạn mức offer càng cao thì conversion càng tăng.

- Các nhóm D2–D5 có tỷ lệ chuyển đổi khá thấp cho cả Cash và Card, trong khi D1 lại có conversion tương đối cao (Cash: 8.09%).

- Từ D6 trở đi, tỷ lệ chuyển đổi tăng mạnh và duy trì ở mức cao:
  - D8: Cash 8.82%, Card 3.50%
  - D9: Cash 8.82%, Card 4.29%
  - D10: Cash 13.04%, Card 6.30%

- D10 là nhóm có hiệu quả cao nhất:
  - Cash: 13.04%
  - Card: 6.30%
  - Uplift: +6.74 điểm %

- Cash Loan vượt Credit Card ở tất cả các decile, cho thấy hạn mức offer cao không làm thay đổi kết luận chung rằng Cash đang hiệu quả hơn Card.

- Nhóm khách hàng được offer hạn mức cao nhất (D8–D10) có tỷ lệ chuyển đổi vượt trội, cho thấy `last_offer_limit` nhiều khả năng đang phản ánh chất lượng khách hàng hoặc mức độ tín nhiệm mà House Credit đã đánh giá trước đó.

- D1 là một điểm bất thường vì conversion cao dù thuộc nhóm hạn mức thấp nhất. Nên kiểm tra thêm:
  - số lượng khách hàng trong D1
  - phân bố `risk_grp`
  - phân bố `cli_age`
  
  để xác định đây là tín hiệu business thực sự hay do cấu trúc dữ liệu.

- `last_offer_limit` là feature có giá trị cho uplift model vì các nhóm hạn mức cao thể hiện mức độ phản hồi mạnh hơn đáng kể đối với cả hai chiến lược.

# 14. Previous Approval vs Conversion

In [ ]:

# Cash treatment -> Cash sign
cash_approve = (
    df[df["treatment"] == "cash"]
    .groupby("f_approve_last_process")["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> Card sign
card_approve = (
    df[df["treatment"] == "card"]
    .groupby("f_approve_last_process")["card_sign_30d"]
    .mean()
    .mul(100)
)

# tạo bảng
approve_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_approve,
    "Card Treatment → Card Sign": card_approve,
})

approve_table["uplift_cash_vs_card"] = (
    approve_table["Cash Treatment → Cash Sign"]
    - approve_table["Card Treatment → Card Sign"]
)

# đổi tên cho dễ đọc
approve_table.index = [
    "Not Approved",
    "Approved"
]

# long format
plot_df = (
    approve_table[
        [
            "Cash Treatment → Cash Sign",
            "Card Treatment → Card Sign"
        ]
    ]
    .reset_index(names="approval_status")
    .melt(
        id_vars="approval_status",
        var_name="treatment",
        value_name="conversion_rate"
    )
)

# vẽ
plt.figure(figsize=(8, 6))

ax = sns.barplot(
    data=plot_df,
    x="approval_status",
    y="conversion_rate",
    hue="treatment"
)

for p in ax.patches:
    height = p.get_height()

    if not np.isnan(height):
        ax.annotate(
            f"{height:.2f}%",
            (
                p.get_x() + p.get_width()/2,
                height
            ),
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.title("Conversion Rate by Previous Approval Status")
plt.xlabel("Previous Approval Status")
plt.ylabel("Conversion Rate (%)")

plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

In [ ]:
approve_table.round(2)

- Khách hàng từng được phê duyệt trước đó có tỷ lệ chuyển đổi cao hơn ở cả hai chiến lược.

- Nhóm Approved:
  - Cash: 8.87%
  - Card: 3.80%
  - Uplift: +5.07 điểm %

- Nhóm Not Approved:
  - Cash: 7.05%
  - Card: 3.54%
  - Uplift: +3.51 điểm %

- Cash Loan tiếp tục vượt Credit Card ở cả hai nhóm, nhưng khoảng cách lớn hơn đáng kể ở nhóm Approved.

- Kết quả cho thấy khách hàng từng được House Credit phê duyệt có xu hướng quay lại sử dụng sản phẩm cao hơn, phản ánh mức độ tin cậy và khả năng tiếp cận tín dụng tốt hơn.

- Chênh lệch uplift giữa hai nhóm:
  - Approved: +5.07%
  - Not Approved: +3.51%
  
  cho thấy chiến lược Cash đặc biệt hiệu quả đối với khách hàng đã từng được phê duyệt.

- `f_approve_last_process` là feature có giá trị cho uplift model vì phản ánh trực tiếp lịch sử đánh giá tín dụng của House Credit và khả năng tái sử dụng sản phẩm của khách hàng.

# 15. Connected Telesales vs Conversion

In [ ]:
df["connected_tls_decile"] = pd.qcut(
    df["cnt_connected_tls"],
    q=10,
    labels=[f"D{i}" for i in range(1, 11)],
    duplicates="drop"
)

cash_tls = (
    df[df["treatment"] == "cash"]
    .groupby("connected_tls_decile")["cash_sign_30d"]
    .mean()
    .mul(100)
)

card_tls = (
    df[df["treatment"] == "card"]
    .groupby("connected_tls_decile")["card_sign_30d"]
    .mean()
    .mul(100)
)

tls_table = pd.DataFrame({
    "Cash Treatment → Cash Sign": cash_tls,
    "Card Treatment → Card Sign": card_tls,
})

tls_table["uplift_cash_vs_card"] = (
    tls_table["Cash Treatment → Cash Sign"]
    - tls_table["Card Treatment → Card Sign"]
)


plot_df = (
    tls_table[
        [
            "Cash Treatment → Cash Sign",
            "Card Treatment → Card Sign"
        ]
    ]
    .reset_index()
    .melt(
        id_vars="connected_tls_decile",
        var_name="treatment",
        value_name="conversion_rate"
    )
)

plt.figure(figsize=(12, 6))

ax = sns.barplot(
    data=plot_df,
    x="connected_tls_decile",
    y="conversion_rate",
    hue="treatment"
)

for p in ax.patches:
    height = p.get_height()

    if not np.isnan(height):
        ax.annotate(
            f"{height:.2f}%",
            (
                p.get_x() + p.get_width()/2,
                height
            ),
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.title("Conversion Rate by Connected Telesales Decile")
plt.xlabel("Connected Telesales Decile")
plt.ylabel("Conversion Rate (%)")

plt.legend(title="Treatment → Outcome")

plt.tight_layout()
plt.show()

In [ ]:
tls_table.round(2)

- Tỷ lệ chuyển đổi tăng rõ rệt từ D1 đến khoảng D4–D5, sau đó gần như đi ngang.

- Cash Loan:
  - D1: 4.99%
  - D5: 6.93%
  - Sau D5 dao động quanh 6%–6.6%

- Credit Card:
  - D1: 2.58%
  - D10: 3.11%
  - Mức tăng tương đối nhỏ hơn Cash Loan.

- Uplift tăng từ +2.41% ở D1 lên +4.04% ở D5, sau đó ổn định quanh mức 3.4%–3.6%.

- Kết quả cho thấy việc tăng số lần kết nối telesales có tác động tích cực ở giai đoạn đầu, nhưng hiệu quả bổ sung giảm dần sau một ngưỡng nhất định.

- Không quan sát thấy xu hướng:
  
```text
càng gọi nhiều → conversion tăng liên tục
```

  mà giống mô hình:

```text
tăng nhanh ban đầu → bão hòa
```

- Điều này gợi ý rằng sau một số lần kết nối thành công, việc tiếp tục gọi thêm có thể không tạo ra nhiều chuyển đổi bổ sung.

- Cash Loan hưởng lợi từ hoạt động telesales nhiều hơn Credit Card vì khoảng cách Cash–Card tăng lên khi mức độ tương tác tăng.

- `cnt_connected_tls` là feature marketing hữu ích cho uplift model vì phản ánh mức độ tương tác thực tế giữa House Credit và khách hàng, đồng thời cho thấy dấu hiệu hiệu ứng bão hòa của telesales.

# 16. Appointment vs Conversion

In [ ]:
# Conversion theo treatment tương ứng
cash_conv = (
    df[df["treatment"] == "cash"]
    .groupby("f_appointment_3m")["cash_sign_30d"]
    .mean()
    .mul(100)
)

card_conv = (
    df[df["treatment"] == "card"]
    .groupby("f_appointment_3m")["card_sign_30d"]
    .mean()
    .mul(100)
)

plot_df = pd.DataFrame({
    "Cash": cash_conv,
    "Card": card_conv
}).reset_index()

plot_df["f_appointment_3m"] = (
    plot_df["f_appointment_3m"]
    .map({
        0: "No Appointment",
        1: "Appointment"
    })
)

# -------------------
# Plot
# -------------------
fig, ax = plt.subplots(figsize=(8,5))

x = np.arange(len(plot_df))
width = 0.35

bars1 = ax.bar(
    x - width/2,
    plot_df["Cash"],
    width,
    label="Cash"
)

bars2 = ax.bar(
    x + width/2,
    plot_df["Card"],
    width,
    label="Card"
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["f_appointment_3m"])

ax.set_ylabel("Conversion Rate (%)")
ax.set_xlabel("Appointment in Last 3 Months")
ax.set_title("Appointment vs Conversion by Treatment")

ax.legend()

for bars in [bars1, bars2]:
    for b in bars:
        h = b.get_height()
        ax.text(
            b.get_x() + b.get_width()/2,
            h,
            f"{h:.2f}%",
            ha="center",
            va="bottom"
        )

plt.tight_layout()
plt.show()

In [ ]:
summary = pd.DataFrame({
    "cash_conv": cash_conv,
    "card_conv": card_conv
})

summary["uplift"] = (
    summary["cash_conv"]
    - summary["card_conv"]
)

summary

- Khách hàng từng đặt lịch hẹn trong 3 tháng gần nhất (`f_appointment_3m = 1`) có tỷ lệ chuyển đổi Cash rất cao (~48.3%), cao gấp hơn 4 lần nhóm chưa từng đặt lịch (~10.9%).

- Conversion của Card gần như không thay đổi giữa hai nhóm (~4.0%–4.5%), cho thấy tín hiệu đặt lịch hẹn chủ yếu liên quan đến nhu cầu vay tiền mặt.

- Uplift tăng mạnh từ 6.35 lên 44.28 điểm %, cho thấy khách đã từng đặt lịch là nhóm "hot lead" cực kỳ tiềm năng cho chiến lược Cash Loan.

- `f_appointment_3m` là một trong những feature business rất mạnh và có khả năng trở thành biến quan trọng trong uplift model.

# 17. Payment Count vs Conversion

In [ ]:
# Tạo decile
df_plot = df.copy()

df_plot["payment_decile"] = pd.qcut(
    df_plot["cnt_total_payment"],
    q=10,
    labels=[f"D{i}" for i in range(1, 11)],
    duplicates="drop"
)

# Cash treatment -> cash_sign_30d
cash_conv = (
    df_plot[df_plot["treatment"] == "cash"]
    .groupby("payment_decile", observed=False)["cash_sign_30d"]
    .mean()
    .mul(100)
)

# Card treatment -> card_sign_30d
card_conv = (
    df_plot[df_plot["treatment"] == "card"]
    .groupby("payment_decile", observed=False)["card_sign_30d"]
    .mean()
    .mul(100)
)

plot_df = pd.DataFrame({
    "Cash": cash_conv,
    "Card": card_conv
})

plot_df["Uplift"] = (
    plot_df["Cash"] - plot_df["Card"]
)

plot_df = plot_df.reset_index()

# ======================
# Plot
# ======================

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(plot_df))
width = 0.35

bars1 = ax.bar(
    x - width/2,
    plot_df["Cash"],
    width,
    label="Cash"
)

bars2 = ax.bar(
    x + width/2,
    plot_df["Card"],
    width,
    label="Card"
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["payment_decile"])

ax.set_xlabel("Total Payment Count Decile")
ax.set_ylabel("Conversion Rate (%)")

ax.set_title(
    "Payment Count vs Conversion by Treatment"
)

ax.legend()

for bars in [bars1, bars2]:
    for b in bars:
        h = b.get_height()
        ax.text(
            b.get_x() + b.get_width()/2,
            h,
            f"{h:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.tight_layout()
plt.show()

In [ ]:
summary = plot_df.copy()

summary["Uplift"] = (
    summary["Cash"]
    - summary["Card"]
)

summary.sort_values(
    "payment_decile"
)

- Conversion tăng gần như đơn điệu từ D1 đến D10 đối với cả Cash và Card, cho thấy số lần thanh toán là tín hiệu rất mạnh về mức độ gắn bó và chất lượng khách hàng.

- Khách hàng có lịch sử thanh toán nhiều hơn có xu hướng chuyển đổi cao hơn rõ rệt. Cash tăng từ 1.51% lên 18.11%, trong khi Card tăng từ 0.99% lên 5.93%.

- Uplift cũng tăng liên tục từ 0.52 lên 12.18 điểm %, cho thấy nhóm khách hàng thanh toán tích cực phản ứng với chiến lược Cash Loan mạnh hơn đáng kể so với Credit Card.

- D9 và D10 là các phân khúc giá trị cao nhất, vừa có conversion cao vừa có uplift lớn, rất phù hợp để ưu tiên chiến dịch Cash Loan.

- `cnt_total_payment` là một feature business rất mạnh và có khả năng nằm trong nhóm predictor quan trọng của uplift model.

# 18. Recency Analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df_plot = df.copy()

# Chia decile theo recency
df_plot["recency_decile"] = pd.qcut(
    df_plot["cnt_month_from_last_transaction"],
    q=10,
    labels=[f"D{i}" for i in range(1, 11)],
    duplicates="drop"
)

# Giá trị trung bình số tháng trong từng decile
month_mapping = (
    df_plot.groupby("recency_decile", observed=False)
    ["cnt_month_from_last_transaction"]
    .mean()
)

# Cash treatment -> cash_sign_30d
cash_conv = (
    df_plot[df_plot["treatment"] == "cash"]
    .groupby("recency_decile", observed=False)
    ["cash_sign_30d"]
    .mean()
    * 100
)

# Card treatment -> card_sign_30d
card_conv = (
    df_plot[df_plot["treatment"] == "card"]
    .groupby("recency_decile", observed=False)
    ["card_sign_30d"]
    .mean()
    * 100
)

plot_df = pd.DataFrame({
    "avg_month_since_last_txn": month_mapping,
    "Cash": cash_conv,
    "Card": card_conv,
})

plot_df = plot_df.sort_values(
    "avg_month_since_last_txn"
).reset_index()

plt.figure(figsize=(10,6))

plt.plot(
    plot_df["avg_month_since_last_txn"],
    plot_df["Cash"],
    marker="o",
    label="Cash"
)

plt.plot(
    plot_df["avg_month_since_last_txn"],
    plot_df["Card"],
    marker="o",
    label="Card"
)

plt.xlabel("Months Since Last Transaction")
plt.ylabel("Conversion Rate (%)")

plt.title(
    "Recency vs Conversion by Treatment"
)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
summary = plot_df.copy()

summary["Uplift"] = (
    summary["Cash"]
    - summary["Card"]
)

summary[
    [
        "avg_month_since_last_txn",
        "Cash",
        "Card",
        "Uplift"
    ]
]

- Conversion có xu hướng giảm khi khoảng thời gian từ giao dịch gần nhất tăng lên, đặc biệt sau mốc khoảng 20 tháng. Nhóm giao dịch gần đây (~2–15 tháng) có conversion cao nhất.

- Cash luôn có conversion cao hơn Card ở mọi nhóm recency, cho thấy lợi thế của chiến lược Cash khá ổn định.

- Uplift đạt mức cao nhất ở khoảng 6–15 tháng kể từ giao dịch gần nhất (khoảng 7.5–8.8 điểm %), sau đó giảm mạnh ở nhóm trên 20 tháng.

- Khách hàng không giao dịch trong thời gian dài (~56 tháng) có conversion thấp nhất cho cả Cash và Card, phản ánh mức độ tương tác với House Credit đã suy giảm đáng kể.

- `cnt_month_from_last_transaction` cho thấy tín hiệu business rất mạnh và nhiều khả năng sẽ là một trong những feature quan trọng của uplift model.

# 19. Loan Termination Behavior

In [ ]:
df_plot = df.copy()

# Bucket hóa
df_plot["termination_bucket"] = pd.cut(
    df_plot["cnt_terminated_loan_12m"],
    bins=[-1, 0, 1, 2, 3, np.inf],
    labels=[
        "0",
        "1",
        "2",
        "3",
        "4+"
    ]
)

# Cash treatment -> cash_sign_30d
cash_conv = (
    df_plot[df_plot["treatment"] == "cash"]
    .groupby("termination_bucket", observed=False)
    ["cash_sign_30d"]
    .mean()
    * 100
)

# Card treatment -> card_sign_30d
card_conv = (
    df_plot[df_plot["treatment"] == "card"]
    .groupby("termination_bucket", observed=False)
    ["card_sign_30d"]
    .mean()
    * 100
)

plot_df = pd.DataFrame({
    "Cash": cash_conv,
    "Card": card_conv
}).reset_index()

plot_df["Uplift"] = (
    plot_df["Cash"]
    - plot_df["Card"]
)

fig, ax = plt.subplots(figsize=(9,5))

x = np.arange(len(plot_df))
width = 0.35

bars1 = ax.bar(
    x - width/2,
    plot_df["Cash"],
    width,
    label="Cash"
)

bars2 = ax.bar(
    x + width/2,
    plot_df["Card"],
    width,
    label="Card"
)

ax.set_xticks(x)
ax.set_xticklabels(
    plot_df["termination_bucket"]
)

ax.set_xlabel(
    "Terminated Loans in Last 12 Months"
)

ax.set_ylabel(
    "Conversion Rate (%)"
)

ax.set_title(
    "Loan Termination Behavior vs Conversion"
)

ax.legend()

for bars in [bars1, bars2]:
    for b in bars:
        h = b.get_height()
        ax.text(
            b.get_x() + b.get_width()/2,
            h,
            f"{h:.1f}%",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.tight_layout()
plt.show()

In [ ]:
summary = plot_df.copy()

summary["Uplift"] = (
    summary["Cash"]
    - summary["Card"]
)

summary

- Conversion tăng rất mạnh theo số lượng khoản vay đã tất toán trong 12 tháng gần nhất. Cash tăng từ 7.46% ở nhóm chưa tất toán lên 26.35% ở nhóm có từ 4 khoản trở lên.

- Card cũng tăng theo xu hướng tương tự nhưng mức tăng thấp hơn đáng kể so với Cash.

- Uplift tăng liên tục từ 4.18 lên 20.10 điểm %, cho thấy khách hàng thường xuyên tất toán khoản vay phản ứng cực kỳ tốt với chiến lược Cash Loan.

- Kết quả này ủng hộ giả thuyết về nhóm **repeat borrower**: khách đã hoàn thành nhiều khoản vay có xu hướng quay lại vay tiếp thay vì chuyển sang sản phẩm thẻ tín dụng.

- `cnt_terminated_loan_12m` là một trong những feature hành vi mạnh nhất đã phân tích đến thời điểm hiện tại và rất có giá trị cho uplift model.

# Import 

In [ ]:
import pandas as pd 
import os 
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
import pickle
import joblib

os.chdir("D:/dataism/round 3.1/src")
os.getcwd()

In [ ]:
df = pd.read_csv("data/cleaned_dazone_sample.csv")
df.head()

# Bỏ các cột có tỉ lệ missing > 50% 

In [ ]:
drop_cols = [
    "f_appointment_3m",
    "max_cntr_len_fi_12m",
    "ratio_monthly_amt_l6m_vs_n6m",
    "ratio_vol_hcvn_cash_appl_12m",
    "cnt_month_from_first_transaction",
    "cnt_month_from_last_transaction",
]

df.drop(columns=drop_cols, inplace=True)
df.columns 

# Bỏ cột id

In [ ]:
drop_cols = [
    "customer_id",
]

df.drop(columns=drop_cols, inplace=True)
df.columns 

# Min max scaler cho các cột numeric 

In [ ]:
num_cont_cols = [
    "amt_terminated_loan_12m",
    "cli_emp_period",
    "last_offer_limit",
    "cnt_month_from_last_appl",
    "cnt_total_payment",
    "cnt_fail_tls_3m",
    "cnt_connected_tls",
    "cnt_bank_inst_appl_12m",
    "cnt_fi_inst_appl_12m",
    "min_sign_vol",
    "ratio_conn_tls_weekend_12m",
    "ratio_conn_tls_endmonth_12m",
    "avg_sign_vol",
    "cli_age",
    "cnt_month_status1",
]

df_num_cont = df[num_cont_cols]
df_num_cont.head()

In [ ]:
imputer = SimpleImputer(
    strategy="median"
)

df_num_cont = pd.DataFrame(
    imputer.fit_transform(df_num_cont),
    columns=df_num_cont.columns,
    index=df_num_cont.index,
)

df_num_cont.isnull().mean() * 100

In [ ]:
scaler = MinMaxScaler()

df_num_cont = pd.DataFrame(
    scaler.fit_transform(df_num_cont),
    columns=df_num_cont.columns,
    index=df_num_cont.index,
)

df_num_cont.describe().T

# Min max scaler các cột numeric cat

In [ ]:
num_cat_cols = [
    "f_approve_last_process",
    "worst_credit_status",
    "f_ever_default",
    "cnt_report_bank_6m",
    "cnt_report_institution_12m",
    "cnt_lender_has2pay_last_24m",
    "cnt_terminated_loan_12m",
    "cnt_month_over3_credit_card_6m",
    "cnt_noinsurance_loan_24m",
    "cnt_earlyterminated_cash",
    "cnt_earlyterminted_pos",
    "cnt_inbound_tls_12m",
]

df_num_cat = df[num_cat_cols]
df_num_cat.head()

In [ ]:
df_num_cat.isnull().mean() * 100

In [ ]:
num_cat_imputer = SimpleImputer(
    strategy="most_frequent"
)

df_num_cat = pd.DataFrame(
    num_cat_imputer.fit_transform(df_num_cat),
    columns=df_num_cat.columns,
    index=df_num_cat.index,
)

df_num_cat.isnull().mean() * 100

In [ ]:
num_cat_scaler = MinMaxScaler()

df_num_cat = pd.DataFrame(
    num_cat_scaler.fit_transform(df_num_cat),
    columns=df_num_cat.columns,
    index=df_num_cat.index,
)

df_num_cat.describe().T

# One hot encoding (bỏ cột đầu tiên) cho các cột nominal 

In [ ]:
nominal_cols = [
    'treatment', 
    'cli_job', 
    'cli_marital_status', 
    'cli_contact_region_level', 
    'last_process_channel', 
    'risk_grp', 
    'cli_contact_region_population_grp'
]

df_nominal = df[nominal_cols]
df_nominal.head()

In [ ]:
df_nominal.isnull().mean() * 100

In [ ]:
nominal_imputer = SimpleImputer(
    strategy="most_frequent"
)

df_nominal = pd.DataFrame(
    nominal_imputer.fit_transform(df_nominal),
    columns=df_nominal.columns,
    index=df_nominal.index,
)

df_nominal.isnull().mean() * 100

In [ ]:
nominal_encoder = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore",
)

encoded = nominal_encoder.fit_transform(df_nominal)

df_nominal = pd.DataFrame(
    encoded,
    columns=nominal_encoder.get_feature_names_out(df_nominal.columns),
    index=df_nominal.index,
)

df_nominal.head()

# Label encoder cho cột ordinal 

In [ ]:
ordinal_cols = [
    'cli_education'
]

df_ordinal = df[ordinal_cols]
df_ordinal.head()

In [ ]:
df_ordinal.isnull().mean() * 100

In [ ]:
ordinal_imputer = SimpleImputer(
    strategy="most_frequent"
)

df_ordinal = pd.DataFrame(
    ordinal_imputer.fit_transform(df_ordinal),
    columns=df_ordinal.columns,
    index=df_ordinal.index,
)

df_ordinal.isnull().mean() * 100

In [ ]:
ordinal_encoder = OrdinalEncoder(
    categories=[
        ['ELEMENTARY_SCHOOL', 'PTCS', 'PTTH',   'BACHELORS', 'MASTERS']       # col_1_A
    ],
    dtype=int,
)

df_ordinal = pd.DataFrame(
    ordinal_encoder.fit_transform(df_ordinal),
    columns=df_ordinal.columns,
    index=df_ordinal.index,
)

df_ordinal.head()

In [ ]:
ordinal_scaler = MinMaxScaler()

df_ordinal = pd.DataFrame(
    ordinal_scaler.fit_transform(df_ordinal),
    columns=df_ordinal.columns,
    index=df_ordinal.index,
)

df_ordinal.describe().T

# Gộp tất cả thành đặc trưng 

In [ ]:
df.head()

In [ ]:
df_feature = pd.concat([df_num_cont, df_num_cat, df_nominal, df_ordinal], axis=1)
df_feature.head()

In [ ]:
df_describe = df_feature.describe().T
print(f"min: {df_describe['min'].unique()}")
print(f"max: {df_describe['max'].unique()}")

# Xem xét 2 cột target

In [ ]:
target_cols = [
    "cash_sign_30d", 
    "card_sign_30d"
]

df_target = df[target_cols]
df_target.head()

In [ ]:
df_target.isnull().mean() * 100

# Lưu dữ liệu

In [ ]:
folder_path = "data/feature_engineering/feature-eng-data-daz/src"
df_feature.to_csv(f"{folder_path}/df_feature.csv", index=False)
df_target.to_csv(f"{folder_path}/df_target.csv", index=False)

# Save các transformer 

In [ ]:
folder_path = "data/feature_engineering/feature-eng-data-daz/src"

transformers = {
    "imputer": imputer,
    "scaler": scaler,
    "num_cat_imputer": num_cat_imputer,
    "num_cat_scaler": num_cat_scaler,
    "nominal_imputer": nominal_imputer,
    "nominal_encoder": nominal_encoder,
    "ordinal_imputer": ordinal_imputer,
    "ordinal_encoder": ordinal_encoder,
    "ordinal_scaler": ordinal_scaler,
}

joblib.dump(transformers, f"{folder_path}/transformers.joblib")